In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm = ChatOpenAI(model="gpt-4o-mini")

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content='네, 잘 지냈어요! 당신은 어떻게 지내고 있나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 12, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f9565ce549', 'id': 'chatcmpl-DWd7LcShhgiKrg0iVxPiWgS7XT3P0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da9bd-17bf-7a21-8077-dc0379e1d51e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 16, 'total_tokens': 28, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [3]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool
def get_current_time(timezone: str, location: str) -> str:
    """
    현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존(예: 'Asia/Seoul'). 실제 존재햐애함
        location (str): 지역명. 타임존은 모든 지명에 대응되지 않으므로 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재 시각 {now}'
    print(location_and_local_time)
    return location_and_local_time

In [4]:
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

llm_with_tools = llm.bind_tools(tools)

In [5]:
from langchain_core.messages import SystemMessage


messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?")
]

response = llm_with_tools.invoke(messages)
messages.append(response)

print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 132, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DWd7MkW2WSFyVZ0JjnCvk56hzOQCV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da9bd-1c72-7731-9f88-bdfa6ac8faa8-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_ILpgbzqTllMWeNayNwIoqCbn', 'type': 'tool_call'}],

In [6]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재 시각 2026-04-20 16:13:57


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 132, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DWd7MkW2WSFyVZ0JjnCvk56hzOQCV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da9bd-1c72-7731-9f88-bdfa6ac8faa8-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_ILpgbzqTllMWeNayNwIoqCbn', 'type': 'tool_call'}

In [7]:
llm_with_tools.invoke(messages)

AIMessage(content='부산은 현재 2026년 4월 20일 16시 13분 57초입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 188, 'total_tokens': 214, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DWd7OPJp06Z4VKfQDproXnFVQahDI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da9bd-212c-7442-a8ff-56431a526202-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 188, 'output_tokens': 26, 'total_tokens': 214, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [8]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")

In [9]:
import yfinance as yf


@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown()

    return history_md

tools = [get_current_time, get_yf_stock_history,]
tool_dict = {
    "get_current_time" : get_current_time,
    "get_yf_stock_history" : get_yf_stock_history,
}

llm_with_tools = llm.bind_tools(tools)

In [10]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐어? 내렸어?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 280, 'total_tokens': 307, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42fcdba006', 'id': 'chatcmpl-DWd7Ptia8Vd5l7td1V9skG93dBBhQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019da9bd-2768-7812-b52e-71742d6c34d0-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'id': 'call_CWuMwkMwAtkJqx9ndXt7V0S8', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 280, 'output_tokens': 27, 'total_tokens': 307, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audi

In [11]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-03-18 00:00:00-04:00 | 399    | 403.07 | 392.31 |  392.78 | 5.08531e+07 |           0 |              0 |\n| 2026-03-19 00:00:00-04:00 | 387.27 | 387.27 | 378.73 |  380.3  | 6.70783e+07 |           0 |              0 |\n| 2026-03-20 00:00:00-04:00 | 379.85 | 379.89 | 364.46 |  367.96 | 7.86286e+07 |           0 |              0 |\n| 2026-03-23 00:00:00-04:00 | 373.09 | 385.33 | 372.73 |  380.85 | 7.4606e+07  |           0 |              0 |\n| 2026-03-24 00:00:00-04:00 | 376.56 | 387.48 | 376.31 |  383.03 | 6.00049e+07 |           0 |              0 |\n| 2026-03-25 00:00:00-04:00 | 389.99 | 396.23 | 385.01 |  385.95 | 5.51573e+07 |           0 |              0 |\n| 2026-03-26 00:00:00-04:0

In [12]:
llm_with_tools.invoke(messages)

AIMessage(content='한 달 전인 2026년 3월 18일 테슬라(TSLA)의 종가는 392.78 달러였고, 현재 시각인 2026년 4월 20일의 종가는 정보가 제공되지 않았습니다. 하지만, 2026년 4월 17일의 종가는 400.62 달러입니다. \n\n따라서, 테슬라의 주가는 한 달 전과 비교하여 오른 것입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 1637, 'total_tokens': 1737, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42fcdba006', 'id': 'chatcmpl-DWd7RzgyQiumKQLOdUeOB67vlOhTq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da9bd-3159-7b83-be31-392b6fde8ba1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1637, 'output_tokens': 100, 'total_tokens': 1737, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_toke

In [15]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점이 무엇인지 이야기해줘.")]):
    print(c.content, end="|")

|안|녕하세요|!| 한국| 사회|의| 문제|점|에| 대한| 이야|기는| 다양|하고| 복|잡|하지만|,| 몇| 가지| 주요| 이|슈|를| 정|리|해|볼| 수| 있습니다|.

|1|.| **|저|출|산| 및| 고|령|화|**|:| 한국|은| 세계|에서| 가장| 낮|은| 출|산|율|을| 기록|하고| 있으며|,| 이는| 장|기|적으로| 인|구| 감소|와| 경제| 성장| 둔|화|의| 요|인이| 될| 수| 있습니다|.| 또한| 고|령|화| 사회|로| 진|입|하면서| 노|인| 복|지|와| 일|자리| 문제|도| 대|두|되고| 있습니다|.

|2|.| **|주|택| 문제|**|:| 특히| 수도|권|의| 집|값| 상승|은| 많은| 사람|들에게| 큰| 부담|이| 되고| 있습니다|.| 주|거| 안정|성이| 떨어|지|면서| 젊|은| 세|대|가| 주|거| 문제|로| 어려|움을| 겪|고| 있습니다|.

|3|.| **|일|자리|의| 질|과| 고|용| 불|안|정|**|:| 비|정|규|직|과| 장|시간| 노동| 문제|는| 여|전히| 해결|되지| 않고| 있으며|,| 청|년|층|에서| 특히| 고|용| 불|안|이| 큰| 문제|로| 인|식|되고| 있습니다|.

|4|.| **|사회|적| 불|평|등|**|:| 소|득| 격|차|와| 사회|적| 계|층| 간|의| 불|평|등|이| 심|화|되고| 있습니다|.| 이는| 교육| 기|회|,| 건강|,| 주|거| 등| 다양한| 분야|에서| 불|균|형|을| 초|래|하고| 있습니다|.

|5|.| **|정|신| 건강| 문제|**|:| 경쟁|이| 치|열|한| 사회| 분위|기|와| 스트|레스| 증가|로| 인해| 정신| 건강| 문제가| 많이| 대|두|되고| 있습니다|.| 특히| 청|년|층|과| 노|인|층|에서| 우|울|증|과| 고|독|감|이| 증가|하고| 있습니다|.

|6|.| **|정|치|적| 양|극|화|**|:| 정치|적| 이|념|이나| 소|속|에| 따른| 갈|등|이| 심|화|되|면서| 사회|적| 갈|등|이| 증가|하고| 있습니다|.| 이는| 사회| 통|합|을| 저|

In [16]:
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 132, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f957560a82', 'id': 'chatcmpl-DWd7MkW2WSFyVZ0JjnCvk56hzOQCV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da9bd-1c72-7731-9f88-bdfa6ac8faa8-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_ILpgbzqTllMWeNayNwIoqCbn', 'type': 'tool_call'}],

In [17]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)

is_first = True

for chunk in response:
    print("chunk type: ", type(chunk))

    if is_first:
        is_first = False
        gathered = chunk

    else:
        gathered += chunk

    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)



chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_25QJyUL6d0vU5jFy3x3fvxCQ', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_25QJyUL6d0vU5jFy3x3fvxCQ', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {}, 'id': 'call_25QJyUL6d0vU5jFy3x3fvxCQ', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': ''}, 'id': 'call_25QJyUL6d0vU5jFy3x3fvxCQ', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:   tool_call_chunk [{'name': 'get_current_time', 'args': {'timezone': 'Asia'}, 'id': 'call_25QJyUL6d0vU5jFy3x3fvxCQ', 'type': 'tool_c

In [18]:
gathered

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42fcdba006', 'service_tier': 'default'}, id='lc_run--019da9c6-eaba-7b10-9ed5-898a43d88d4c', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_25QJyUL6d0vU5jFy3x3fvxCQ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 200, 'output_tokens': 23, 'total_tokens': 223, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_25QJyUL6d0vU5jFy3x3fvxCQ', 'index': 0, 'type': 'tool_call_chunk'}], chunk_position='last')

In [19]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

messages

{'timezone': 'Asia/Seoul', 'location': '부산'}
Asia/Seoul (부산) 현재 시각 2026-04-20 16:26:09


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai', 'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42fcdba006', 'service_tier': 'default'}, id='lc_run--019da9c6-eaba-7b10-9ed5-898a43d88d4c', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul', 'location': '부산'}, 'id': 'call_25QJyUL6d0vU5jFy3x3fvxCQ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 200, 'output_tokens': 23, 'total_tokens': 223, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, tool_call_chunks=[{'name': 'get_current_time', 'args': '{"timezone":"Asia/Seoul","location":"부산"}', 'id': 'call_25QJyUL6d0vU5jFy3x3fvxCQ', 'index': 0, 'ty

In [20]:
for c in llm_with_tools.stream(messages):
    print(c.content, end="|")

|부|산|의| 현재| 시|각|은| |202|6|년| |4|월| |20|일| |16|시| |26|분|입니다|.||||